In [1]:
import pandas as pd
import re
from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
amazon_labelled = pd.read_csv('sentiment labelled sentences\\amazon_cells_labelled.txt', sep='\t', names=['text', 'label'])
print(amazon_labelled.shape)
print(amazon_labelled.head())

(1000, 2)
                                                text  label
0  So there is no way for me to plug it in here i...      0
1                        Good case, Excellent value.      1
2                             Great for the jawbone.      1
3  Tied to charger for conversations lasting more...      0
4                                  The mic is great.      1


In [3]:
amazon_labelled['source'] = 'amazon'
print(amazon_labelled.head())

                                                text  label  source
0  So there is no way for me to plug it in here i...      0  amazon
1                        Good case, Excellent value.      1  amazon
2                             Great for the jawbone.      1  amazon
3  Tied to charger for conversations lasting more...      0  amazon
4                                  The mic is great.      1  amazon


In [4]:
imdb_labelled = pd.read_csv('sentiment labelled sentences\\imdb_labelled.txt', sep='\t', names=['text', 'label'], quoting=3)
print(imdb_labelled.shape)
print(imdb_labelled.head())
print(imdb_labelled.label.value_counts())

(1000, 2)
                                                text  label
0  A very, very, very slow-moving, aimless movie ...      0
1  Not sure who was more lost - the flat characte...      0
2  Attempting artiness with black & white and cle...      0
3       Very little music or anything to speak of.        0
4  The best scene in the movie was when Gerardo i...      1
label
0    500
1    500
Name: count, dtype: int64


In [5]:
imdb_labelled['source'] = 'imdb'
print(imdb_labelled.head())

                                                text  label source
0  A very, very, very slow-moving, aimless movie ...      0   imdb
1  Not sure who was more lost - the flat characte...      0   imdb
2  Attempting artiness with black & white and cle...      0   imdb
3       Very little music or anything to speak of.        0   imdb
4  The best scene in the movie was when Gerardo i...      1   imdb


In [6]:
yelp_labelled = pd.read_csv('sentiment labelled sentences\\yelp_labelled.txt', sep='\t', names=['text', 'label'])
print(yelp_labelled.shape)
print(yelp_labelled.head())

(1000, 2)
                                                text  label
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1


In [7]:
yelp_labelled['source'] = 'yelp'
print(yelp_labelled.head())

                                                text  label source
0                           Wow... Loved this place.      1   yelp
1                                 Crust is not good.      0   yelp
2          Not tasty and the texture was just nasty.      0   yelp
3  Stopped by during the late May bank holiday of...      1   yelp
4  The selection on the menu was great and so wer...      1   yelp


In [8]:
data = pd.concat([amazon_labelled, imdb_labelled, yelp_labelled], ignore_index=True)

In [9]:
print('shape', data.shape)
print('count of NAN:\n', data.isna().sum())
print('distribution\n', data.groupby(['source', 'label']).size())

shape (3000, 3)
count of NAN:
 text      0
label     0
source    0
dtype: int64
distribution
 source  label
amazon  0        500
        1        500
imdb    0        500
        1        500
yelp    0        500
        1        500
dtype: int64


In [10]:
duplicates_text = data.duplicated(subset='text')
duplicates_all = data.duplicated(subset=['text', 'label'])
text_indexes = list(duplicates_text[duplicates_text == True].index)
all_indexes = list(duplicates_all[duplicates_all == True].index)

print(text_indexes == all_indexes)

True


In [11]:
data = data.drop_duplicates(subset=['text'], ignore_index=True)

print('shape', data.shape)
print(f'distribution\n{data.groupby(['source', 'label']).size()}')

shape (2983, 3)
distribution
source  label
amazon  0        497
        1        493
imdb    0        499
        1        498
yelp    0        497
        1        499
dtype: int64


In [12]:
stratify_tt = data.source + '_' + data.label.astype('str')
X_trainval, X_test, y_trainval, y_test = train_test_split(
    data.text, data.label,
    test_size=0.15,
    random_state=42,
    stratify=stratify_tt
)


stratify_tv = data.loc[X_trainval.index].source + '_' + data.loc[X_trainval.index].label.astype('str')
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.18,
    random_state=42,
    stratify=stratify_tv
)

print(f'X_train shape: {X_train.shape}\nX_val shape: {X_val.shape}\nX_test shape: {X_test.shape}')
print(data.loc[X_test.index].groupby(['source', 'label']).size())
print(data.loc[X_train.index].groupby(['source', 'label']).size())
print(data.loc[X_val.index].groupby(['source', 'label']).size())

X_train shape: (2078,)
X_val shape: (457,)
X_test shape: (448,)
source  label
amazon  0        75
        1        74
imdb    0        75
        1        75
yelp    0        74
        1        75
dtype: int64
source  label
amazon  0        346
        1        343
imdb    0        348
        1        347
yelp    0        347
        1        347
dtype: int64
source  label
amazon  0        76
        1        76
imdb    0        76
        1        76
yelp    0        76
        1        77
dtype: int64


Разделение сохранило состав данных по источникам и классам.

In [13]:
def tokenize(text: str) -> list:
    text = text.lower()
    text = text.replace('’', "'")
    tokens = re.findall(r'[a-zа-яё]+(?:[\'][а-яёa-z]+)*|\.|\,|\?|\!', text)

    return tokens

In [14]:
train_tokens = X_train.apply(tokenize)
print("все отзывы токенизировались" if len(train_tokens) == X_train.shape[0] else 'не все отзывы токенизировались')
print('индексы сохранились' if train_tokens.index.equals(X_train.index) else 'индексы не сохранились')
print(sum(1 for l in train_tokens if len(l) == 0))
print(train_tokens)

все отзывы токенизировались
индексы сохранились
0
2039                 [their, chow, mein, is, so, good, !]
2482                      [and, then, tragedy, struck, .]
1136    [the, fish, is, badly, made, and, some, of, it...
2169     [i, dressed, up, to, be, treated, so, rudely, !]
1650    [however, paul, schrader, has, indeed, made, a...
                              ...                        
1868    [i, was, bored, throughout, the, whole, damn, ...
2556                        [i, consider, this, theft, .]
1804    [you, will, leave, the, theater, wanting, to, ...
2264    [the, goat, taco, didn't, skimp, on, the, meat...
1338                                [it's, just, lame, .]
Name: text, Length: 2078, dtype: object


In [15]:
def get_tokens_frequency(list_of_tokens):
    total = [l[w] for l in list_of_tokens for w in range(len(l))]
    tokens_freq = Counter(t for t in total)

    return tokens_freq


train_tokens_freq = get_tokens_frequency(train_tokens)
ten_most_common = train_tokens_freq.most_common(10)
print(*ten_most_common, sep='\n')

('.', 2125)
('the', 1335)
(',', 913)
('and', 811)
('i', 637)
('a', 613)
('is', 509)
('to', 479)
('it', 478)
('this', 467)


**Сравнение двух вариантов без построения словаря**: сколько обычных токенов останется при min_freq=1 и при min_freq=2

In [16]:
total = [l[w] for l in train_tokens for w in range(len(l))]
l1, l2 = [], []
for k, v in train_tokens_freq.items():
    if v < 1:
        continue
    l1.append(v)

for k, v in train_tokens_freq.items():
    if v < 2:
        continue
    l2.append(v)

skip_1 = len(total) - sum(l1)
skip_2 = len(total) - sum(l2)

print(f"Всего {len(total)} слов")
print(f'При min_freq=1 мы пропустим {skip_1} токена(-ов), останется {len(l1)} уникальных токена(-ов).')
print(f'При min_freq=2 пропустим {skip_2} токена(-ов), останется {len(l2)} уникальных токена(-ов).')
print(f'При min_freq=1 будет заменено {skip_1 / len(total) * 100:0.1f}% слов.')
print(f'При min_freq=2 будет заменено {skip_2 / len(total) * 100:0.1f}% слов.')

Всего 28235 слов
При min_freq=1 мы пропустим 0 токена(-ов), останется 4193 уникальных токена(-ов).
При min_freq=2 пропустим 2426 токена(-ов), останется 1767 уникальных токена(-ов).
При min_freq=1 будет заменено 0.0% слов.
При min_freq=2 будет заменено 8.6% слов.


In [17]:
def make_dict(tokens_freq: dict, min_freq = 1):
    dictionary = {}
    dictionary.setdefault('<PAD>', 0)
    dictionary.setdefault('<UNK>', 1)
    i = 2
    for k, v in tokens_freq.items():
        if v < min_freq:
            continue
        dictionary.setdefault(k, i)
        i += 1

    return dictionary


dictionary = make_dict(train_tokens_freq)
print(len(dictionary))
print('Is max index == (len(dictionary) - 1)?', max(dictionary.values()) == (len(dictionary) - 1))
for k, v in dictionary.items():
    print(f'({k}, {v})')
    if v == 10:
        break

4195
Is max index == (len(dictionary) - 1)? True
(<PAD>, 0)
(<UNK>, 1)
(their, 2)
(chow, 3)
(mein, 4)
(is, 5)
(so, 6)
(good, 7)
(!, 8)
(and, 9)
(then, 10)


In [18]:
def tokens_to_ids(tokens: list, dictionary) -> list:
    ids = []
    for token in tokens:
        ind = dictionary.get(token, dictionary['<UNK>'])
        ids.append(ind)
    return ids

In [19]:
def check_encoder(encode_tokens, dictionary):
    dictionary_before = dictionary.copy()

    known = next(t for t in dictionary if t not in ("<PAD>", "<UNK>"))
    unknown = "__несуществующий_токен__"
    while unknown in dictionary:
        unknown += "_"

    tokens = [known, unknown, known]
    tokens_before = tokens.copy()

    assert encode_tokens([], dictionary) == [], \
        "Пустой список должен давать []"

    result = encode_tokens(tokens, dictionary)

    assert isinstance(result, list), "Результат должен быть списком"
    assert result == [
        dictionary[known],
        dictionary["<UNK>"],
        dictionary[known],
    ], "Нарушены индексы, порядок или обработка неизвестного токена"

    assert tokens == tokens_before, "Функция изменила входной список"
    assert dictionary == dictionary_before, "Функция изменила словарь"

    print("Все проверки пройдены")


check_encoder(tokens_to_ids, dictionary)

Все проверки пройдены


In [20]:
train_ids = []
for l in train_tokens:
    train_ids.append(tokens_to_ids(l, dictionary=dictionary))

In [21]:
val_tokens = X_val.apply(tokenize)
val_ids = []
for l in val_tokens:
    val_ids.append(tokens_to_ids(l, dictionary))